# A1 Bear Camera AI — ONNX エクスポート (Colab)

このノートブックは Raspberry Pi 上の NCNN 推理が segfault する環境でも、
`models/yolo_bear.onnx` を作ることで cv2.dnn (OpenCV) ベースの安定推理に切り替えるための、
Pi 側へのインストール不要のエクスポート手順です。

## 流れ

1. **CPU ランタイムを選択**（GPU でも可だが CPU 推奨）
2. リポジトリの `best.pt`（または `models/yolo_bear.pt`）を Colab にアップロード。
3. torch==2.5.1+cpu + ultralytics + onnx + onnxscript をインストール。
4. `yolo_bear.onnx` を opset 18 でエクスポート → opset 12 に変換（imgsz 256, simplify）。
5. ダウンロードして Raspberry Pi の `models/yolo_bear.onnx` に配置。

> **2026-07 更新**: Colab のデフォルト Python 3.12 + torch 2.11 では
> onnxscript が `torch_2_11` サブモジュールを欠いておりエクスポート失敗する。
> そのため torch を 2.5.1 にダウングレードする（cp312 wheel あり）。
> torch 2.5+ の dynamo ベース ONNX exporter は opset 18 以上しか出力できないが、
> エクスポート後に `onnx.version_converter` で opset 12 に変換することで
> Raspberry Pi の cv2.dnn (OpenCV 4.x) と互換性を保つ。

安全性境界：この Camera AI はあくまで追加の感知レイヤであり、Arduino Uno Q の
接触パッド状態機械やフェイルセーフ RELEASE_OFF 動作を置き換えるものではない。

In [1]:
import sys
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [ ]:
# Colab のデフォルト Python 3.12 + torch 2.11 では onnxscript が
# torch_2_11 サブモジュールを欠いているため ONNX エクスポートに失敗する。
# → torch 2.5.1 にダウングレード（cp312 wheel あり、onnxscript 互換あり）。
# opset 18 でエクスポート後、バージョン変換で opset 12 に下げる。
%pip uninstall -y torch torchvision torchaudio onnx onnxscript onnxslim ultralytics
%pip install -q torch==2.5.1 --index-url https://download.pytorch.org/whl/cpu
%pip install -q ultralytics==8.3.40 onnx==1.16.1 onnxslim onnxscript

Found existing installation: torch 2.12.1+cpu
Uninstalling torch-2.12.1+cpu:
  Successfully uninstalled torch-2.12.1+cpu
Found existing installation: torchvision 0.27.1
Uninstalling torchvision-0.27.1:
  Successfully uninstalled torchvision-0.27.1
Found existing installation: onnx 1.16.1
Uninstalling onnx-1.16.1:
  Successfully uninstalled onnx-1.16.1
Found existing installation: onnxscript 0.5.7
Uninstalling onnxscript-0.5.7:
  Successfully uninstalled onnxscript-0.5.7
Found existing installation: onnxslim 0.1.94
Uninstalling onnxslim-0.1.94:
  Successfully uninstalled onnxslim-0.1.94
Found existing installation: ultralytics 8.3.40
Uninstalling ultralytics-8.3.40:
  Successfully uninstalled ultralytics-8.3.40
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.6/174.6 MB 7.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 69.3 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are instal

In [ ]:
import torch
import ultralytics
print('ultralytics', ultralytics.__version__)
print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())

## 1. best.pt をアップロード

Colab のファイルブラウザまたは次のセルで `best.pt` を `/content/best.pt` として用意する。

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()  # best.pt を選択
if not uploaded:
    raise RuntimeError('best.pt がアップロードされていません。')
name = next(iter(uploaded))
src = Path('/content') / name
dst = Path('/content/best.pt')
if src != dst:
    src.rename(dst)
print('model:', dst, dst.stat().st_size, 'bytes')

## 2. ONNX エクスポート

imgsz は学習と同じ 256。NCNN と同じ入力サイズを維持する。

> **2026-07 更新**: torch==2.5.1 の dynamo ベース ONNX exporter を使う。
> opset 18 でエクスポートし、後続セルで opset 12 への変換を試みる
> （Raspberry Pi cv2.dnn 互換用）。変換失敗時は onnxruntime を使用。

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/best.pt', task='detect')
print('classes:', model.names)

# torch 2.5+ の dynamo exporter は opset >= 18 必須。
# 後続セルで opset 12 へ変換する。
export_path = model.export(
    format='onnx',
    imgsz=256,
    opset=18,
    simplify=True,
    dynamic=False,
    half=False,
    batch=1,
)
print('exported_onnx:', export_path)

In [ ]:
# opset 18 → 12 変換（Raspberry Pi cv2.dnn 互換用）
# モダン torch は opset 18 以上しか出力できないため、後処理でダウングレードする

import onnx
from onnx import version_converter
from pathlib import Path

m = onnx.load(export_path)
original_opset = m.opset_import[0].version
print(f'original opset: {original_opset}')

try:
    m12 = version_converter.convert_version(m, 12)
    export_path_12 = str(Path(export_path).with_suffix('.opset12.onnx'))
    onnx.save(m12, export_path_12)
    print(f'opset 12 converted: {export_path_12}')
    # 以降のセルで使うパスを opset 12 版に差し替え
    export_path = export_path_12
except Exception as e:
    print(f'opset conversion failed: {e}')
    print('opset 18 のまま進みます。Raspberry Pi では onnxruntime を使用してください。')
    print('pip install onnxruntime で対応可能です。')

In [ ]:
import onnx
from pathlib import Path

export_path_str = export_path if isinstance(export_path, str) else str(export_path)
m = onnx.load(export_path_str)
opset = m.opset_import[0].version
print(f'file: {export_path_str}')
print(f'opset: {opset}')
print('inputs:', [(i.name, [d.dim_value for d in i.type.tensor_type.shape.dim]) for i in m.graph.input])
print('outputs:', [(o.name, [d.dim_value for d in o.type.tensor_type.shape.dim]) for o in m.graph.output])
assert opset <= 15, f'opset {opset} > 15, Raspberry Pi cv2.dnn 非対応。onnxruntime を使用してください。'
print('✅ opset <= 15, cv2.dnn 互換あり')

## 3. yolo_bear.onnx にリネームしてダウンロード

出力を `yolo_bear.onnx` にしてダウンロード。Raspberry Pi 上で
`models/yolo_bear.onnx` として配置すると、Camera AI は自動で
ONNX/cv2.dnn ベースの推理に切り替わる（NCNN ディレクトリより優先）。

> opset 12 変換済みのファイルがダウンロードされる。
> 変換に失敗した場合でも opset 18 版がダウンロードされるので、
> その場合は Pi 側で `pip install onnxruntime` して推論バックエンドを切り替える。

In [ ]:
from pathlib import Path
from google.colab import files

src = Path(export_path) if isinstance(export_path, str) else Path(str(export_path))
final = Path('/content/yolo_bear.onnx')
if src != final:
    src.rename(final)
print('final:', final, final.stat().st_size, 'bytes')
files.download(str(final))

## 4. Raspberry Pi 側での配置

ダウンロードした `yolo_bear.onnx` を Pi 上の
`2026_Hackathon/models/yolo_bear.onnx` に置く。

```bash
# Pi 側
ls -la models/yolo_bear.onnx
```

その後、`./scripts/run_demo.sh` を起動すると、Camera AI は
ONNX/cv2.dnn 推理で動き、`ai_model_ok=true` となる。
NCNN が segfault 環境でも PyTorch 不要で安定動作する。

> **opset 変換失敗時の対応**: 変換に失敗して opset 18 のままの場合は、
> Pi 側で `pip install onnxruntime` を実行し、Camera AI の推論バックエンドを
> cv2.dnn から onnxruntime に切り替えること。
> onnxruntime は opset 18 以上をネイティブサポートしている。